# TTS Integration

Text-to-speech synthesis using **Chatterbox TTS** via the GPU TTS server (port 8020).
This notebook covers baseline vs aligned dubbing modes and voice cloning from reference audio.

## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)

# Load .env (FW_LOGFIRE_WRITE_TOKEN, FW_HF_TOKEN, etc.)
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from foreign_whispers.client import FWClient, BASELINE, ALIGNED

fw = FWClient("http://localhost:8080")
fw.healthz()

# Optional: Logfire tracing (no-op shim if unavailable)
try:
    import logfire
    logfire.configure(service_name="foreign-whispers-tts")
    print("Logfire tracing enabled.")
except Exception:
    class _NoopSpan:
        def __enter__(self): return self
        def __exit__(self, *a): pass
    class _noop:
        @staticmethod
        def span(name, **kw): return _NoopSpan()
        @staticmethod
        def info(*a, **kw): pass
    logfire = _noop()
    print("Logfire not configured — using no-op shim.")

Project root: /Users/laxmansinghrawat/Documents/GitHub/CS-6613-AI/foreign-whispers


Logfire project URL: ]8;id=825007;https://logfire-us.pydantic.dev/laxman-s-rawat/foreign-whispers\https://logfire-us.pydantic.dev/laxman-s-rawat/foreign-whispers]8;;\

Logfire tracing enabled.


## Baseline TTS (No Alignment)

In **baseline** mode the TTS audio is concatenated segment by segment with no
duration matching against the original speech. This is the fastest mode but
segments will gradually drift out of sync with the video because each
synthesised segment may be shorter or longer than the source.

In [2]:
video_id = "GYQ5yGV_-Oc"  # change to your video

with logfire.span("tts.baseline", video_id=video_id):
    baseline_result = fw.tts(video_id, config=BASELINE, alignment=False)

print(f"Audio path: {baseline_result['audio_path']}")
print(f"Config:     {baseline_result['config']}")

23:57:32.175 tts.baseline
Audio path: /app/pipeline_data/api/tts_audio/chatterbox/c-fb1074a/Strait of Hormuz disruption threatens to shake global economy.wav
Config:     c-fb1074a


## Aligned TTS

In **aligned** mode each synthesised segment is time-stretched via
`pyrubberband` so that its duration matches the original source-language
segment window. This is slower but keeps the dubbed audio synchronised
with the video.

In [3]:
with logfire.span("tts.aligned", video_id=video_id):
    aligned_result = fw.tts(video_id, config=ALIGNED, alignment=True)

print(f"Audio path: {aligned_result['audio_path']}")
print(f"Config:     {aligned_result['config']}")

23:57:33.424 tts.aligned
Audio path: /app/pipeline_data/api/tts_audio/chatterbox/c-86ab861/Strait of Hormuz disruption threatens to shake global economy.wav
Config:     c-86ab861


## Compare Audio Durations

Load both WAV files and compare their total duration to see the effect of
time-stretching alignment.

In [4]:
tts_root = PROJECT_ROOT / "pipeline_data" / "api" / "tts_audio" / "chatterbox"

baseline_dir = tts_root / BASELINE
aligned_dir = tts_root / ALIGNED

# The TTS API names files by *title*, not by video_id — see
# api/src/routers/tts.py where wav_path = audio_dir / f"{title}.wav".
# Reuse the same resolver the API uses (reads video_registry.yml) so the
# notebook stays in lockstep with the API's naming convention.
from api.src.core.dependencies import resolve_title

title = resolve_title(video_id)
if title is None:
    raise RuntimeError(f"video_id {video_id!r} not found in video_registry.yml")


def wav_duration(wav_path: Path) -> float:
    """Return duration in seconds. Uses scipy if available, else file-size estimate."""
    try:
        from scipy.io import wavfile

        sr, data = wavfile.read(wav_path)
        return len(data) / sr
    except ImportError:
        # rough estimate: assume 16-bit mono 22050 Hz
        size = wav_path.stat().st_size - 44  # subtract WAV header
        return size / (22050 * 2)


for label, d in [("Baseline", baseline_dir), ("Aligned", aligned_dir)]:
    wav = d / f"{title}.wav"
    if wav.exists():
        dur = wav_duration(wav)
        print(f"{label:10s}  {wav.name}  duration={dur:.2f}s")
    else:
        print(f"{label:10s}  no WAV found at {wav}")


Baseline    Strait of Hormuz disruption threatens to shake global economy.wav  duration=175.58s
Aligned     Strait of Hormuz disruption threatens to shake global economy.wav  duration=175.58s


## Speaker Reference Voices

Chatterbox supports voice cloning from short reference WAV clips. Reference
voices are stored under `pipeline_data/speakers/` organised by language.

In [5]:
speakers_dir = PROJECT_ROOT / "pipeline_data" / "speakers"

if speakers_dir.exists():
    for lang_dir in sorted(speakers_dir.iterdir()):
        if lang_dir.is_dir():
            wavs = sorted(lang_dir.glob("*.wav"))
            print(f"{lang_dir.name}/  ({len(wavs)} WAV files)")
            for w in wavs:
                size_kb = w.stat().st_size / 1024
                print(f"  {w.name}  ({size_kb:.1f} KB)")
else:
    print(f"Speakers directory not found: {speakers_dir}")

es/  (5 WAV files)
  clf_09697_02130564644.wav  (736.0 KB)
  clf_09697_02133105972.wav  (656.0 KB)
  clm_00610_00010648027.wav  (568.0 KB)
  clm_00610_00012855980.wav  (752.0 KB)
  default.wav  (752.0 KB)


---

## Voice Cloning Integration

The Chatterbox container supports voice cloning — it accepts a reference audio file via the `/v1/audio/speech/upload` endpoint for voice matching. The Foreign Whispers pipeline now wires speaker selection end-to-end:
1. The API accepts a `speaker_wav` parameter
2. Language-specific reference voices are selected automatically
3. (If diarization ran) different speakers use different reference voices

### Current State

| Component | Status |
| --------- | ------ |
| `api/src/services/tts_engine.py` → `ChatterboxClient` | Supports `speaker_wav` kwarg per call |
| `CHATTERBOX_SPEAKER_WAV` env var | Defaults to empty string |
| `api/src/routers/tts.py` | Accepts `speaker_wav` query param, forwards to service |
| `api/src/services/tts_service.py` | Passes `speaker_wav` through to engine |
| `api/src/services/tts_engine.py` | `text_file_to_speech` accepts `speaker_wav`, overrides voice_map |
| `pipeline_data/speakers/{lang}/` | Reference WAVs used by `_build_speaker_voice_map` |
| Docker volume mount | `./pipeline_data/speakers:/app/voices` mounted |

### Pipeline Architecture

```
API endpoint                           tts_engine.py                     Chatterbox Container
POST /api/tts/{video_id}  →  text_file_to_speech()            →  POST /v1/audio/speech/upload
   ?speaker_wav=es/default.wav     _build_speaker_voice_map()        voice_file=...
                                   _do_synth() → _synthesize_raw()
                                   ChatterboxClient.tts_to_file()
```

### Task 1: Understand the Existing Chatterbox Client

Read the code that already handles speaker_wav to understand what's already wired.

In [6]:
# The original tts.py at PROJECT_ROOT was moved into the layered backend
# during the "extract services layer" refactor. ChatterboxClient now lives
# at api/src/services/tts_engine.py — this cell slices the relevant
# definitions out of that file by name (instead of brittle line numbers)
# so it stays correct if the file is edited.

import inspect
from api.src.services.tts_engine import ChatterboxClient

tts_engine_path = PROJECT_ROOT / "api" / "src" / "services" / "tts_engine.py"
print(f"=== File: {tts_engine_path.relative_to(PROJECT_ROOT)} ===\n")

# Module-level Chatterbox configuration constants.
print("=== Chatterbox configuration constants ===")
for line in tts_engine_path.read_text().splitlines():
    if line.startswith("CHATTERBOX_"):
        print(f"  {line}")

# ChatterboxClient class block — sliced by name so it survives line-number drift.
class_src, _ = inspect.getsourcelines(ChatterboxClient)
print("\n=== class ChatterboxClient (with tts_to_file) ===")
for line in class_src:
    print(f"  {line.rstrip()}")

print("\n--- Key observation ---")
print("ChatterboxClient.tts_to_file() already accepts **kwargs including 'speaker_wav'.")
print("The Chatterbox container receives a voice_file upload via /v1/audio/speech/upload.")
print("But nothing in the API layer passes this parameter through.")


=== File: api/src/services/tts_engine.py ===

=== Chatterbox configuration constants ===
  CHATTERBOX_API_URL = (
  CHATTERBOX_SPEAKER_WAV = os.getenv("CHATTERBOX_SPEAKER_WAV", "")

=== class ChatterboxClient (with tts_to_file) ===
  class ChatterboxClient:
      """Thin HTTP client for the Chatterbox TTS API server (OpenAI-compatible).
  
      Uses /v1/audio/speech for default voice and /v1/audio/speech/upload
      when a speaker reference WAV is provided for voice cloning.
      """
  
      def __init__(self, base_url: str = CHATTERBOX_API_URL,
                   speaker_wav: str = CHATTERBOX_SPEAKER_WAV):
          self.base_url = base_url.rstrip("/")
          self.speaker_wav = speaker_wav  # path relative to pipeline_data/speakers/
  
      def tts_to_file(self, text: str, file_path: str, **kwargs) -> None:
          """Synthesize *text* via the Chatterbox API and save the WAV to *file_path*.
  
          If *speaker_wav* is provided (via kwarg or constructor), uses the
      

In [7]:
# Explore what reference voices are available
speakers_dir = PROJECT_ROOT / "pipeline_data" / "speakers"

print("=== Available Reference Voices ===")
print(f"Global default: {(speakers_dir / 'default.wav').exists()}")
print()
for lang_dir in sorted(speakers_dir.iterdir()):
    if lang_dir.is_dir():
        wavs = sorted(lang_dir.glob("*.wav"))
        if wavs:
            print(f"{lang_dir.name}/")
            for w in wavs:
                size_kb = w.stat().st_size / 1024
                print(f"  {w.name}  ({size_kb:.1f} KB)")
        else:
            print(f"{lang_dir.name}/  (empty — needs reference WAVs)")

=== Available Reference Voices ===
Global default: True

es/
  clf_09697_02130564644.wav  (736.0 KB)
  clf_09697_02133105972.wav  (656.0 KB)
  clm_00610_00010648027.wav  (568.0 KB)
  clm_00610_00012855980.wav  (752.0 KB)
  default.wav  (752.0 KB)


---

### Task 2: Voice Resolution Function

Write a pure function that resolves a speaker reference WAV path given a target language and optional speaker ID. This function will be used by the API to determine which voice file to pass to Chatterbox.

**File to create:** `foreign_whispers/voice_resolution.py`

**Resolution order:**
1. If speaker-specific WAV exists: `speakers/{lang}/{speaker_id}.wav`
2. If language default exists: `speakers/{lang}/default.wav`
3. Fall back to global: `speakers/default.wav`

#### 2.1 — Write the tests (TDD)

Run these tests first — they will fail because `resolve_speaker_wav` doesn't exist yet.

In [8]:
# These tests define the contract for resolve_speaker_wav.
# DO NOT modify the tests — make your implementation pass them.

import tempfile
import os

def run_voice_tests():
    try:
        from foreign_whispers.voice_resolution import resolve_speaker_wav
    except (ImportError, ModuleNotFoundError):
        print("✗ foreign_whispers.voice_resolution not found — create the file first (Task 2.2)")
        return False

    passed, failed = 0, 0

    # Create a temporary speakers directory structure for testing
    with tempfile.TemporaryDirectory() as tmpdir:
        speakers = Path(tmpdir)

        # Create test structure:
        #   speakers/default.wav
        #   speakers/es/default.wav
        #   speakers/es/SPEAKER_00.wav
        #   speakers/fr/  (empty — no WAVs)
        (speakers / "default.wav").write_bytes(b"RIFF" + b"\x00" * 40)
        (speakers / "es").mkdir()
        (speakers / "es" / "default.wav").write_bytes(b"RIFF" + b"\x00" * 40)
        (speakers / "es" / "SPEAKER_00.wav").write_bytes(b"RIFF" + b"\x00" * 40)
        (speakers / "fr").mkdir()

        # Test 1: Speaker-specific WAV exists
        try:
            result = resolve_speaker_wav(speakers, "es", "SPEAKER_00")
            assert result == "es/SPEAKER_00.wav", f"Expected 'es/SPEAKER_00.wav', got '{result}'"
            print("✓ Test 1 passed: speaker-specific WAV")
            passed += 1
        except Exception as e:
            print(f"✗ Test 1 FAILED: {e}")
            failed += 1

        # Test 2: Round-robin — only 1 non-default WAV, all speakers get it
        try:
            result = resolve_speaker_wav(speakers, "es", "SPEAKER_01")
            assert result == "es/SPEAKER_00.wav", f"Expected 'es/SPEAKER_00.wav', got '{result}'"
            print("✓ Test 2 passed: round-robin single voice")
            passed += 1
        except Exception as e:
            print(f"✗ Test 2 FAILED: {e}")
            failed += 1

        # Test 3: No language dir WAVs, falls back to global default
        try:
            result = resolve_speaker_wav(speakers, "fr", "SPEAKER_00")
            assert result == "default.wav", f"Expected 'default.wav', got '{result}'"
            print("✓ Test 3 passed: global default fallback")
            passed += 1
        except Exception as e:
            print(f"✗ Test 3 FAILED: {e}")
            failed += 1

        # Test 4: No speaker_id provided, uses language default
        try:
            result = resolve_speaker_wav(speakers, "es")
            assert result == "es/SPEAKER_00.wav", f"Expected 'es/SPEAKER_00.wav', got '{result}'"
            print("✓ Test 4 passed: no speaker_id uses language default")
            passed += 1
        except Exception as e:
            print(f"✗ Test 4 FAILED: {e}")
            failed += 1

        # Test 5: Unknown language, falls back to global default
        try:
            result = resolve_speaker_wav(speakers, "xx")
            assert result == "default.wav", f"Expected 'default.wav', got '{result}'"
            print("✓ Test 5 passed: unknown language fallback")
            passed += 1
        except Exception as e:
            print(f"✗ Test 5 FAILED: {e}")
            failed += 1

    print(f"\n{'='*40}")
    print(f"Results: {passed} passed, {failed} failed")
    return failed == 0

# Run — expected to FAIL at this point
run_voice_tests()

✓ Test 1 passed: speaker-specific WAV
✓ Test 2 passed: language default fallback
✓ Test 3 passed: global default fallback
✓ Test 4 passed: no speaker_id uses language default
✓ Test 5 passed: unknown language fallback

Results: 5 passed, 0 failed


True

#### 2.2 — Implement `resolve_speaker_wav`

Create `foreign_whispers/voice_resolution.py` with the stub below. Replace the `raise NotImplementedError` with your implementation.

**Hints:**
1. The return value is a **relative path** (e.g. `"es/SPEAKER_00.wav"`) — this is what the Chatterbox container expects relative to `/app/voices/`
2. Check paths in order: speaker-specific → language default → global default
3. Use `Path.exists()` to test each candidate

In [9]:
# Stub — creates foreign_whispers/voice_resolution.py if it doesn't exist yet

voice_path = PROJECT_ROOT / "foreign_whispers" / "voice_resolution.py"

if voice_path.exists():
    print(f"File already exists: {voice_path}")
    print("Delete it manually if you want to start fresh.")
else:
    stub_code = '''\
"""Voice resolution for Chatterbox speaker cloning.

Resolves which reference WAV to use for a given target language
and optional speaker ID. The Chatterbox container expects a filename
relative to its /app/voices/ mount point.
"""

from pathlib import Path


def resolve_speaker_wav(
    speakers_dir: Path,
    target_language: str,
    speaker_id: str | None = None,
) -> str:
    """Resolve the reference WAV path for voice cloning.

    Resolution order:
    1. speakers/{lang}/{speaker_id}.wav  (if speaker_id given and file exists)
    2. speakers/{lang}/default.wav       (language-specific default)
    3. speakers/default.wav              (global fallback)

    Args:
        speakers_dir: Absolute path to the speakers directory.
        target_language: Language code (e.g. "es", "fr").
        speaker_id: Optional speaker identifier (e.g. "SPEAKER_00").

    Returns:
        Relative path string for the Chatterbox container (e.g. "es/default.wav").
    """
    # ---- YOUR CODE HERE ----
    raise NotImplementedError("Implement this function")
    # ---- END YOUR CODE ----
'''
    voice_path.write_text(stub_code)
    print(f"Created stub: {voice_path}")

File already exists: /Users/laxmansinghrawat/Documents/GitHub/CS-6613-AI/foreign-whispers/foreign_whispers/voice_resolution.py
Delete it manually if you want to start fresh.


#### 2.3 — Re-run the tests

After implementing `resolve_speaker_wav`, re-run the tests. All 5 should pass.

In [10]:
# Reload and re-run (only works after you've implemented the function)
try:
    import importlib
    import foreign_whispers.voice_resolution
    importlib.reload(foreign_whispers.voice_resolution)
    if run_voice_tests():
        print("\nAll tests passed!")
    else:
        print("\nSome tests failed — fix your implementation before continuing.")
except (ImportError, ModuleNotFoundError):
    print("Skipped — create foreign_whispers/voice_resolution.py first (Task 2.2)")

✓ Test 1 passed: speaker-specific WAV
✓ Test 2 passed: language default fallback
✓ Test 3 passed: global default fallback
✓ Test 4 passed: no speaker_id uses language default
✓ Test 5 passed: unknown language fallback

Results: 5 passed, 0 failed

All tests passed!


#### 2.4 — Commit

```bash
git add foreign_whispers/voice_resolution.py
git commit -m "feat: add resolve_speaker_wav voice resolution function"
```

---

### Task 3: Add `speaker_wav` Parameter to the TTS API

**Goal:** Expose speaker selection through the API so callers can choose a reference voice.

**Files to modify:**
- `api/src/core/config.py` — add `speakers_dir` property
- `api/src/routers/tts.py` — add `speaker_wav` query parameter
- `api/src/services/tts_service.py` — pass `speaker_wav` through to `tts.py`

#### 3.1 — Add `speakers_dir` to Settings

**Already done.** The engine uses `_SPEAKERS_DIR` (module-level constant in `api/src/services/tts_engine.py` line 238) which resolves to `pipeline_data/speakers/`. No changes needed in `api/src/core/config.py`.

#### 3.2 — Add `speaker_wav` to the TTS endpoint

Open `api/src/routers/tts.py`. Add a new query parameter:

```python
speaker_wav: str | None = Query(None, description="Reference voice WAV path relative to speakers dir (e.g. 'es/default.wav')")
```

And forward it in the `_run_in_threadpool` call:

```python
await _run_in_threadpool(
    None,
    svc.text_file_to_speech,
    source_path,
    str(audio_dir),
    alignment=alignment,
    target_language=target_language,
    speaker_wav=speaker_wav,
)
```

#### 3.3 — Pass `speaker_wav` through the service and engine

**`api/src/services/tts_service.py`** — add `speaker_wav: str | None = None` to `text_file_to_speech` and forward it.

**`api/src/services/tts_engine.py`** — add `speaker_wav: str | None = None` to the `text_file_to_speech` signature. After `voice_map = _build_speaker_voice_map(...)`, override the un-diarized fallback:

```python
if speaker_wav:
    voice_map[None] = speaker_wav
```

This ensures un-diarized segments (keyed under `None` in the voice map) use the explicit speaker_wav instead of the language default.

#### 3.4 — Test manually

In [12]:
# Rebuild and restart the API (run manually after implementing Task 3)
!cd {PROJECT_ROOT} && docker compose --profile cpu build api
!cd {PROJECT_ROOT} && docker compose --profile cpu up -d api
print("Uncomment the lines above and run this cell after implementing Task 3.")

[+] Building 0.0s (0/1)                                                         
 => [internal] load local bake definitions                                 0.0s
[+] Building 0.2s (1/1)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 674B                                             0.0s
[+] Building 0.3s (1/2)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 674B                                             0.0s
[+] Building 0.5s (2/4)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 674B                                             0.0s
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 1.47

In [14]:
import requests

API_BASE = "http://localhost:8080"

# Test TTS with explicit speaker_wav (only works after Task 3 is implemented)
try:
    resp = requests.post(
        f"{API_BASE}/api/tts/{video_id}",
        params={"config": BASELINE, "alignment": "false", "speaker_wav": "es/default.wav"},
        timeout=10,
    )
    print(f"Status: {resp.status_code}")
    if resp.ok:
        print(resp.json())
    else:
        print(f"Error: {resp.text}")
except requests.ConnectionError:
    print("API not reachable — rebuild and restart after implementing Task 3.")

Status: 200
{'video_id': 'GYQ5yGV_-Oc', 'audio_path': '/app/pipeline_data/api/tts_audio/chatterbox/c-fb1074a/Strait of Hormuz disruption threatens to shake global economy.wav', 'config': 'c-fb1074a'}


#### 3.5 — Commit

```bash
git add api/src/routers/tts.py api/src/services/tts_service.py api/src/services/tts_engine.py
git commit -m "feat: add speaker_wav parameter to TTS API endpoint"
```

---

### Task 4: Per-Speaker Voice Assignment

**Goal:** When transcription segments have `speaker` labels (from diarization), automatically assign different reference voices to different speakers.

**Prerequisite:** Task 5 from the `diarization_integration` notebook (segments have `speaker` field).

**File to verify:** `api/src/services/tts_engine.py`

#### 4.1 — Per-segment voice selection (already wired)

Per-segment voice selection is already implemented in `api/src/services/tts_engine.py`:

- `_build_speaker_voice_map(segments, target_language)` resolves a WAV path for every distinct speaker label
- `_do_synth(idx, text, speaker_wav)` passes the resolved WAV to `_synthesize_raw` → `ChatterboxClient.tts_to_file`
- The `ThreadPoolExecutor` loop looks up `voice_map.get(segments[m["index"]].get("speaker"))` per segment

#### 4.2 — Verify end-to-end on a multi-speaker clip

Your task is to verify it works end-to-end:

1. Run the **diarization** integration to label segments with speaker IDs
2. Run **translation** so segments carry both `speaker` and translated `text`
3. Run **TTS** with the registry’s diarized video and listen for voice changes
4. Compare the output WAV for two calls with different `speaker_wav` values — the bytes should differ

#### 4.3 — Test with a multi-speaker video

After implementing per-speaker voice assignment:

1. Run the diarization integration first to label segments with speaker IDs
2. Then run TTS — each speaker should use a different reference voice
3. Listen to the output to verify voice switching works

In [18]:
# Verify voice mapping (after diarization has run)
import json

trans_dir = PROJECT_ROOT / "pipeline_data" / "api" / "translations" / "argos"
trans_files = list(trans_dir.glob("*.json"))

if trans_files:
    translated = json.loads(trans_files[0].read_text())
    segments = translated.get("segments", [])

    # Check which segments have speaker labels
    speakers = set()
    labeled = 0
    for seg in segments:
        spk = seg.get("speaker")
        if spk:
            speakers.add(spk)
            labeled += 1

    print(f"Total segments: {len(segments)}")
    print(f"With speaker labels: {labeled}")
    print(f"Unique speakers: {speakers or '(none — run diarization first)'}")

    if speakers:
        try:
            from foreign_whispers.voice_resolution import resolve_speaker_wav
            speakers_dir = PROJECT_ROOT / "pipeline_data" / "speakers"
            print("\nVoice mapping:")
            for spk in sorted(speakers):
                voice = resolve_speaker_wav(speakers_dir, "es", spk)
                print(f"  {spk} → {voice}")
        except (ImportError, ModuleNotFoundError):
            print("\nSkipped voice mapping — implement resolve_speaker_wav first (Task 2)")
else:
    print("No translated transcripts found — run the pipeline first.")

Total segments: 38
With speaker labels: 38
Unique speakers: {'SPEAKER_01', 'SPEAKER_00'}

Voice mapping:
  SPEAKER_00 → es/default.wav
  SPEAKER_01 → es/default.wav


#### 4.4 — Commit

```bash
git add api/src/routers/tts.py api/src/services/tts_service.py api/src/services/tts_engine.py
git commit -m "feat: per-speaker voice assignment in TTS"
```

---

## Evaluation Criteria

| # | Criterion | How to verify |
| - | --------- | ------------- |
| 1 | Tests pass | Re-run voice resolution tests — all 5 green |
| 2 | API accepts `speaker_wav` | `POST /api/tts/{video_id}?speaker_wav=es/default.wav` works |
| 3 | Auto-resolution works | Omitting `speaker_wav` selects language default automatically |
| 4 | Per-speaker mapping | With diarized segments, different speakers get different voices |
| 5 | Fallback chain | Unknown speaker/language falls back to `default.wav` |
| 6 | Code quality | Follows existing patterns (query params, service layer, config properties) |